In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Activation, Conv2D, Flatten, Input, Reshape, AveragePooling2D, MaxPooling2D, Conv2DTranspose, TimeDistributed, LSTM, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.regularizers import l2
import time

import random 

I0000 00:00:1792255054.170635     377 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
data_path = os.path.expanduser('~/climatebench/data/')

**data loading**

In [3]:
scenarios = ['ssp126','ssp370','ssp585','hist-GHG', 'hist-aer']
SECONDS_PER_DAY = 86400
len_historical = 165
slider = 10 #ten year window

In [4]:
X_train = []
Y_train = []

for i, scenario in enumerate(scenarios):
    input_name = 'inputs_' + scenario + '.nc'
    output_name = 'outputs_' + scenario + '.nc'

    #historical only for hist-aer and hist-ghg data
    if 'hist' in scenario:
        # load inputs 
        input_xr = xr.open_dataset(data_path + input_name)
            
        # load outputs                                                             
        output_xr = xr.open_dataset(data_path + output_name).mean(dim='member')
        #convert to mm/day
        output_xr = output_xr.assign({"pr": output_xr.pr * SECONDS_PER_DAY,
                                      "pr90": output_xr.pr90 * SECONDS_PER_DAY}).rename({'lon':'longitude', 
                                                                               'lat': 'latitude'}).transpose('time','latitude', 'longitude').drop_vars('quantile')
    #historical combined with ssps
    else:
        # load inputs 
        input_xr = xr.open_mfdataset([data_path + 'inputs_historical.nc', 
                                    data_path + input_name]).compute()
            
        # load outputs                                                             
        output_xr = xr.concat([xr.open_dataset(data_path + 'outputs_historical.nc').mean(dim='member'),
                               xr.open_dataset(data_path + output_name).mean(dim='member')],
                               dim='time').compute()
        #convert to mm/day
        output_xr = output_xr.assign({"pr": output_xr.pr * SECONDS_PER_DAY,
                                      "pr90": output_xr.pr90 * SECONDS_PER_DAY}).rename({'lon':'longitude', 
                                                                               'lat': 'latitude'}).transpose('time','latitude', 'longitude').drop_vars('quantile')

    print(input_xr.dims, scenario)

    #add to training data
    X_train.append(input_xr)
    Y_train.append(output_xr)

FrozenMappingWarningOnValuesAccess({'time': 251, 'latitude': 96, 'longitude': 144}) ssp126
FrozenMappingWarningOnValuesAccess({'time': 251, 'latitude': 96, 'longitude': 144}) ssp370
FrozenMappingWarningOnValuesAccess({'time': 251, 'latitude': 96, 'longitude': 144}) ssp585
FrozenMappingWarningOnValuesAccess({'time': 165, 'latitude': 96, 'longitude': 144}) hist-GHG
FrozenMappingWarningOnValuesAccess({'time': 165, 'latitude': 96, 'longitude': 144}) hist-aer


**normalize data**

In [5]:
#helper fxns
def normalize(data, var, meanstd_dict):
    mean = meanstd_dict[var][0]
    std = meanstd_dict[var][1]
    return (data - mean)/std

def unnormalize(data, var, meanstd_dict):
    mean = meanstd_dict[var][0]
    std = meanstd_dict[var][1]
    return data * std + mean

In [6]:
#get mean/std of inputs for normalization 
meanstd_inputs = {}

for var in ['CO2', 'CH4', 'SO2', 'BC']:
    # To not take the historical data into account several time we have to slice the scenario datasets
    # and only keep the historical data once (in the first ssp index 0 in the simus list)
    array = np.concatenate([X_train[i][var].data for i in [0, 3, 4]] + 
                           [X_train[i][var].sel(time=slice(len_historical, None)).data for i in range(1, 3)])
    print((array.mean(), array.std()))
    meanstd_inputs[var] = (array.mean(), array.std())

#normalize based on calculated mean/std
X_train_norm = [] 
for i, train_xr in enumerate(X_train): 
    for var in ['CO2', 'CH4', 'SO2', 'BC']: 
        var_dims = train_xr[var].dims
        train_xr=train_xr.assign({var: (var_dims, normalize(train_xr[var].data, var, meanstd_inputs))}) 
    X_train_norm.append(train_xr)

(np.float64(1074.172303244536), np.float64(1755.690699230666))
(np.float64(0.1927369743762821), np.float64(0.18457590641432994))
(np.float64(2.562335999706673e-12), np.float64(2.2501145667832657e-11))
(np.float64(1.494790500981809e-13), np.float64(1.0313342554838398e-12))


**reshaping data**

In [7]:
# Functions for reshaping the data --> splits data into overlapping 10 year windows
def input_for_training(X_train_xr, skip_historical=False, len_historical=None): 
    
    #turns input into one numpy arr
    X_train_np =  X_train_xr.to_array().transpose('time', 'latitude', 'longitude', 'variable').data

    time_length = X_train_np.shape[0]
    # If we skip historical data, the first sequence created has as last element the first scenario data point
    if skip_historical:
        X_train_to_return = np.array([X_train_np[i:i+slider] for i in range(len_historical-slider+1, time_length-slider+1)])
    # Else we just go through the whole dataset historical + scenario (does not matter in the case of 'hist-GHG' and 'hist_aer')
    else:
        X_train_to_return = np.array([X_train_np[i:i+slider] for i in range(0, time_length-slider+1)])
    
    return X_train_to_return 


def output_for_training(Y_train_xr, var, skip_historical=False, len_historical=None): 
    Y_train_np = Y_train_xr[var].data
    
    time_length = Y_train_np.shape[0]
    
    # If we skip historical data, the first sequence created has as target element the first scenario data point
    if skip_historical:
        Y_train_to_return = np.array([[Y_train_np[i+slider-1]] for i in range(len_historical-slider+1, time_length-slider+1)])
    # Else we just go through the whole dataset historical + scenario (does not matter in the case of 'hist-GHG' and 'hist_aer')
    else:
        Y_train_to_return = np.array([[Y_train_np[i+slider-1]] for i in range(0, time_length-slider+1)])
    
    return Y_train_to_return


**build test set**

In [8]:
vars_to_predict = ['tas', 'diurnal_temperature_range', 'pr', 'pr90']

# Open and reformat test data 
X_test = xr.open_mfdataset([data_path + 'inputs_historical.nc',
                            data_path + 'inputs_ssp245.nc']).compute()

# Normalize input data 
for var in ['CO2', 'CH4', 'SO2', 'BC']: 
    var_dims = X_test[var].dims
    X_test = X_test.assign({var: (var_dims, normalize(X_test[var].data, var, meanstd_inputs))}) 
    
X_test_np = input_for_training(X_test, skip_historical=False, len_historical=len_historical)  

**build and train model**

In [9]:
results_dir = os.path.expanduser('~/climatebench/xiang_week3/cnn_results/')
os.makedirs(results_dir, exist_ok=True)

truth = xr.open_dataset(data_path + 'outputs_ssp245.nc').mean('member')
truth = truth.sel(time=slice(2015, 2100)).compute()
truth['pr']   = truth['pr'] * 86400      # mm/day, same units as training targets
truth['pr90'] = truth['pr90'] * 86400

def get_rmse(true, pred):
    weights = np.cos(np.deg2rad(true.lat))
    return float(np.sqrt(((true - pred) ** 2).weighted(weights).mean(['lat', 'lon'])).mean())

test_years = X_test.time.data[slider-1:]
keep = (test_years >= 2015) & (test_years <= 2100)

In [10]:
SEEDS = [0, 1, 2, 3, 4]
results = []

# inputs are the same for every variable, so build them once
X_train_all = np.concatenate([input_for_training(X_train_norm[i], skip_historical=(i<2), len_historical=len_historical)
                              for i in range(len(scenarios))], axis=0).astype(np.float32)
print("X_train_all:", X_train_all.shape)

for var_to_predict in vars_to_predict:
    Y_train_all = np.concatenate([output_for_training(Y_train[i], var_to_predict, skip_historical=(i<2), len_historical=len_historical)
                                  for i in range(len(scenarios))], axis=0).astype(np.float32)

    for seed in SEEDS:
        pred_file = os.path.join(results_dir, f"pred_{var_to_predict}_seed{seed}.npy")

        if os.path.exists(pred_file):                      # finished in an earlier run
            pred_np = np.load(pred_file)
            print(f"{var_to_predict} seed {seed}: loaded")
        else:
            t0 = time.time()
            keras.backend.clear_session()
            keras.utils.set_random_seed(seed)

            cnn_model = Sequential()
            cnn_model.add(Input(shape=(slider, 96, 144, 4)))
            cnn_model.add(TimeDistributed(Conv2D(20, (3, 3), padding='same', activation='relu')))
            cnn_model.add(TimeDistributed(AveragePooling2D(2)))
            cnn_model.add(TimeDistributed(GlobalAveragePooling2D()))
            cnn_model.add(LSTM(25, activation='relu'))
            cnn_model.add(Dense(1*96*144))
            cnn_model.add(Activation('linear'))
            cnn_model.add(Reshape((1, 96, 144)))
            cnn_model.compile(optimizer="rmsprop", loss="mse", metrics=["mse"])

            cnn_model.fit(X_train_all, Y_train_all, batch_size=16, epochs=30, verbose=2)

            pred_np = cnn_model.predict(X_test_np, verbose=0)[:, 0][keep]   # (86, 96, 144), mm/day for precip
            np.save(pred_file, pred_np)
            print(f"{var_to_predict} seed {seed}: done in {(time.time()-t0)/60:.1f} min")

        pred = xr.DataArray(pred_np, coords=truth[var_to_predict].coords, dims=truth[var_to_predict].dims)
        true = truth[var_to_predict]
        results.append({
            "variable": var_to_predict, "seed": seed,
            "rmse_all_years": get_rmse(true, pred),
            "rmse_2081_2100_mean": get_rmse(true.sel(time=slice(2081, 2100)).mean('time'),
                                            pred.sel(time=slice(2081, 2100)).mean('time')),
        })

    del Y_train_all

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(results_dir, "seed_results.csv"), index=False)
results_df

X_train_all: (726, 10, 96, 144, 4)
tas seed 0: loaded
tas seed 1: loaded
tas seed 2: loaded
tas seed 3: loaded
tas seed 4: loaded
diurnal_temperature_range seed 0: loaded
diurnal_temperature_range seed 1: loaded
diurnal_temperature_range seed 2: loaded
diurnal_temperature_range seed 3: loaded
diurnal_temperature_range seed 4: loaded
pr seed 0: loaded


E0000 00:00:1792255070.107603     377 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/30
46/46 - 62s - 1s/step - loss: 0.3224 - mse: 0.3224
Epoch 2/30
46/46 - 60s - 1s/step - loss: 0.3027 - mse: 0.3027
Epoch 3/30
46/46 - 59s - 1s/step - loss: 0.2859 - mse: 0.2859
Epoch 4/30
46/46 - 58s - 1s/step - loss: 0.2824 - mse: 0.2824
Epoch 5/30
46/46 - 59s - 1s/step - loss: 0.2810 - mse: 0.2810
Epoch 6/30
46/46 - 59s - 1s/step - loss: 0.2802 - mse: 0.2802
Epoch 7/30
46/46 - 59s - 1s/step - loss: 0.2799 - mse: 0.2799
Epoch 8/30
46/46 - 60s - 1s/step - loss: 0.2795 - mse: 0.2795
Epoch 9/30
46/46 - 58s - 1s/step - loss: 0.2792 - mse: 0.2792
Epoch 10/30
46/46 - 59s - 1s/step - loss: 0.2787 - mse: 0.2787
Epoch 11/30
46/46 - 59s - 1s/step - loss: 0.2781 - mse: 0.2781
Epoch 12/30
46/46 - 59s - 1s/step - loss: 0.2772 - mse: 0.2772
Epoch 13/30
46/46 - 58s - 1s/step - loss: 0.2759 - mse: 0.2759
Epoch 14/30
46/46 - 60s - 1s/step - loss: 0.2742 - mse: 0.2742
Epoch 15/30
46/46 - 59s - 1s/step - loss: 0.2730 - mse: 0.2730
Epoch 16/30
46/46 - 59s - 1s/step - loss: 0.2719 - mse: 0.2719
E

,variable,seed,rmse_all_years,rmse_2081_2100_mean
0,tas,0,0.457227,0.420239
1,tas,1,0.372316,0.232932
2,tas,2,0.408300,0.239361
3,tas,3,0.384110,0.270592
4,tas,4,0.371841,0.218440
5,diurnal_temperature_range,0,0.144930,0.096197
6,diurnal_temperature_range,1,0.144256,0.092893
7,diurnal_temperature_range,2,0.143966,0.088867
8,diurnal_temperature_range,3,0.144848,0.095413
9,diurnal_temperature_range,4,0.145585,0.097261


In [11]:
summary = (results_df.groupby("variable")[["rmse_all_years", "rmse_2081_2100_mean"]]
           .agg(["mean", "std", "min", "max"]).reindex(vars_to_predict))
summary.to_csv(os.path.join(results_dir, "seed_summary.csv"))
print(summary.round(4))

units = {"tas": "K", "diurnal_temperature_range": "K", "pr": "mm/day", "pr90": "mm/day"}
for v in vars_to_predict:
    m, s = summary.loc[v, ("rmse_2081_2100_mean", "mean")], summary.loc[v, ("rmse_2081_2100_mean", "std")]
    print(f"{v}: {m:.3f} ± {s:.3f} {units[v]} (2081–2100 mean RMSE, {len(SEEDS)} seeds)")

                          rmse_all_years                          \
                                    mean     std     min     max   
variable                                                           
tas                               0.3988  0.0359  0.3718  0.4572   
diurnal_temperature_range         0.1447  0.0006  0.1440  0.1456   
pr                                0.5172  0.0009  0.5165  0.5186   
pr90                              1.5135  0.0030  1.5092  1.5161   

                          rmse_2081_2100_mean                          
                                         mean     std     min     max  
variable                                                               
tas                                    0.2763  0.0827  0.2184  0.4202  
diurnal_temperature_range              0.0941  0.0034  0.0889  0.0973  
pr                                     0.1529  0.0071  0.1421  0.1613  
pr90                                   0.4799  0.0477  0.4305  0.5574  
tas: 0.276 ± 0.083 